# Step 3 — Present value of the top firms: Precision Agriculture

**Sector:** Precision Agriculture  **Reference company:** John Deere

Each selected firm is scored on four dimensions, computed on DOCDB patent families:

| Dimension | Indicator | Reference |
|---|---|---|
| Volume | firm families ÷ all sector families in the window | Ernst (1998) |
| Influence | forward citations within 5 years, self-citations excluded, ÷ mean of the same CPC subclass × priority year | Hall, Jaffe & Trajtenberg (2001, 2005) |
| Success | granted ÷ filed national applications, per family, averaged over the firm's families | Guellec & van Pottelsberghe (2000) |
| Globalization | mean number of distinct markets (US, CA, EU, BR, CN, JP) per family | Harhoff, Scherer & Vopel (2003); markets from the Step 2 PEST |

The structure follows the course notebook `40_present_value_score.ipynb`. The applicant names come from the Step 2 harmonisation.
Sections 1-7 compute the four indicators. Section 8 combines them under several weighting schemes. The base case is equal weights; the reasoning is in the document "Step 3 – Present-value score: weighting decision (team 38)".

## 0. Before running: export from Step 2

This notebook reads the harmonised applicants of Step 2. Add this cell at the **end of the Step 2 notebook**, run it once, and check that the two files exist in `outputs/`:

```python
OUT = PROJECT_DIR / "outputs"
OUT.mkdir(exist_ok=True)
applicant_links.to_csv(OUT / "step2_applicant_links.csv", index=False)          # unit_id, year, applicant
applicant_summary.reset_index().to_csv(OUT / "step2_applicant_summary.csv", index=False)   # applicant, families, type, country
```

## 1. Setup and parameters

In [ ]:
import re
import sys
import yaml
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sqlalchemy import func

from epo.tipdata.patstat import PatstatClient
from epo.tipdata.patstat.database.models import (
    TLS201_APPLN,
    TLS224_APPLN_CPC,
    TLS206_PERSON,
    TLS207_PERS_APPLN,
    TLS228_DOCDB_FAM_CITN,
)

pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 80)

NAVY, BLUE, ORANGE, GREY = "#0b2a4a", "#3b73b9", "#f08a00", "#5b6570"

In [ ]:
PROJECT_DIR = Path("/home/jovyan/team-38-PrecisionAgriculture")
CONFIG_PATH = PROJECT_DIR / "config" / "sector_PA_FINAL.yaml"
with CONFIG_PATH.open("r", encoding="utf-8") as file:
    config = yaml.safe_load(file)

SRC_DIR = PROJECT_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
from pa_family_selection import build_selected_families

OUT = PROJECT_DIR / "outputs"
OUT.mkdir(exist_ok=True)

SECTOR_NAME = "Precision Agriculture"
WINDOW = (2010, 2017)      # priority years that are scored
CIT_YEARS = 5              # forward-citation window, in years
EXCLUDE_SELF = True        # drop citations between families of the same firm
LAST_YEAR = 2025           # last year of citations covered by the PATSTAT edition: CHECK on TIP
N_FIRMS = 15               # firms to score at most
MIN_FAMILIES = 20          # minimum portfolio in the window: below it the rates are noise
MIN_COHORT = 5             # minimum families in a CPC subclass x year cell, otherwise the year alone is used

patstat = PatstatClient(env=config["patstat"]["environment"])
db = patstat.orm()
print(f"Window {WINDOW[0]}-{WINDOW[1]}, citations counted within {CIT_YEARS} years, up to {LAST_YEAR}")

### Why this window

The Step 2 sector runs from 2000 to 2022. Grants and citations need time: a family filed in 2021 has had no time to be granted or cited, and would look weak only because it is young. We score the families with priority years **2010-2017**. The grant rate of the sector is stable at 0.70-0.73 up to 2016, falls slightly in 2017 (0.68) and clearly in 2018 (0.62): applications of 2018 are still being examined, so we stop the window in 2017. All the sector families of the same years are used as the denominator of Volume and as the reference of the citation normalisation.

## 2. Families and applicants

In [ ]:
selected_families = build_selected_families(db, config)
fam = patstat.df(db.query(selected_families.c.fam_id.label("docdb_family_id"),
                          selected_families.c.family_year.label("priority_year")))
fam = fam.astype({"docdb_family_id": int, "priority_year": int})
print(f"Step 0 selection: {len(fam):,} families")

families = fam[fam.priority_year.between(*WINDOW)].copy().reset_index(drop=True)
FAMILIES = [int(f) for f in families.docdb_family_id]
print(f"In the window {WINDOW[0]}-{WINDOW[1]}: {len(FAMILIES):,} families")
assert len(fam) == 5181, "The Step 0 population should be 5,181 families"

In [ ]:
links = pd.read_csv(OUT / "step2_applicant_links.csv")
summary = pd.read_csv(OUT / "step2_applicant_summary.csv").set_index("applicant")
print(f"{len(links):,} family-applicant links, {links.applicant.nunique():,} harmonised applicants")
summary.head(10)

### 2.1 Groups: acquisitions completed within the window

Step 2 keeps legal entities of the same group separate. For the score we merge into one firm the applicants whose acquisition was **completed by the end of the window (2017)**. Acquisitions completed later are left separate.

| Merged into | Applicants | Completed | Source |
|---|---|---|---|
| MONSANTO | CLIMATE (The Climate Corporation) | Oct 2013, into Monsanto | Wikipedia, The Climate Corporation |
| AGCO | PRECISION PLANTING | 5 Sept 2017, from The Climate Corporation | Seed World, 5 Sept 2017 |
| not merged | BAYER and MONSANTO | 7 June 2018, after the end of the window | Bayer press release, 7 June 2018 |
| not merged | RAVEN INDUSTRIES | acquired by CNH on 30 Nov 2021 | CNH press release, 30 Nov 2021 |

The merge is applied to all years of the window, as in the course notebook: a firm is scored as the group it belongs to at the end of the window. The cell prints every label that is merged: **check the list**, a pattern can catch a name it should not.

In [ ]:
GROUPS = {
    "MONSANTO": r"^MONSANTO\b|^CLIMATE\b",      # Bayer joins only in June 2018: to merge it, use "BAYER": r"^BAYER\b|^MONSANTO\b|^CLIMATE\b"
    "AGCO": r"^AGCO\b|^PRECISION PLANTING\b",
}

def firm_name(label):
    for group, pattern in GROUPS.items():
        if re.search(pattern, label):
            return group
    return label

links["firm"] = links.applicant.map(firm_name)
merged = links[links.firm != links.applicant]
display(merged.groupby(["firm", "applicant"]).unit_id.nunique().rename("families, 2000-2022").to_frame())

### 2.2 Which firms do we score?

Only applicants of type *firm* are scored (universities and research bodies are left out). A family filed by two firms together counts for both. The selection takes the firms with the most families in the window, with a minimum portfolio.

In [ ]:
firm_types = summary["type"].reindex(links.applicant).values
links["type"] = firm_types
is_firm = links.type.eq("firm") | links.firm.isin(GROUPS)      # a merged group is a firm

fam_firm_all = links[is_firm][["unit_id", "firm"]].drop_duplicates().rename(columns={"unit_id": "docdb_family_id"})
fam_firm = fam_firm_all[fam_firm_all.docdb_family_id.isin(FAMILIES)]

n_families = fam_firm.groupby("firm").docdb_family_id.nunique().sort_values(ascending=False)
TOP = list(n_families[n_families >= MIN_FAMILIES].head(N_FIRMS).index)
SHORT = {f: f.title()[:28] for f in TOP}

print(f"{len(TOP)} firms scored. Firms above the gate: {(n_families >= MIN_FAMILIES).sum()}")
pd.concat([n_families.loc[TOP].rename("families in window"),
           fam_firm_all.groupby("firm").docdb_family_id.nunique().reindex(TOP).rename("families 2000-2022")], axis=1)

In [ ]:
def per_firm(column, how="mean"):
    """Aggregate a column of `families` over the families of each scored firm."""
    table = fam_firm.merge(families[["docdb_family_id", column]], on="docdb_family_id")
    return table.groupby("firm")[column].agg(how).reindex(TOP).rename(column)


def firm_bars(values, title, xlabel, fmt="{:.2f}", ref=None, ref_label=None, color=NAVY):
    v = values.reindex(TOP)[::-1]
    fig, ax = plt.subplots(figsize=(9, 0.42 * len(v) + 1.2))
    ax.barh([SHORT[f] for f in v.index], v.values, color=color)
    for y, x in enumerate(v.values):
        if pd.notna(x):
            ax.text(x, y, " " + fmt.format(x), va="center", ha="left", fontsize=9, color=GREY)
    if ref is not None:
        ax.axvline(ref, color=ORANGE, ls="--", lw=1.2)
        ax.text(ref, len(v) - 0.45, f" {ref_label}", color=ORANGE, fontsize=9, va="bottom")
    ax.set_xlabel(xlabel)
    ax.set_title(title)
    plt.tight_layout()
    plt.show()

## 3. Data of the families: first publication year and CPC subclasses

In [ ]:
q = (db.query(TLS201_APPLN.docdb_family_id, func.min(TLS201_APPLN.earliest_publn_year).label("first_publn_year"))
     .filter(TLS201_APPLN.docdb_family_id.in_(FAMILIES))
     .filter(TLS201_APPLN.earliest_publn_year < 9999)
     .group_by(TLS201_APPLN.docdb_family_id))
families = families.merge(patstat.df(q), on="docdb_family_id", how="left")

q = (db.query(TLS201_APPLN.docdb_family_id,
              func.replace(TLS224_APPLN_CPC.cpc_class_symbol, " ", "").label("cpc"))
     .join(TLS224_APPLN_CPC, TLS224_APPLN_CPC.appln_id == TLS201_APPLN.appln_id)
     .filter(TLS201_APPLN.docdb_family_id.in_(FAMILIES))
     .distinct())
family_cpc = patstat.df(q)
family_cpc["subclass"] = family_cpc.cpc.str[:4]
family_cpc = family_cpc[["docdb_family_id", "subclass"]].drop_duplicates()

print(f"Families without first publication year: {families.first_publn_year.isna().sum()}")
print(f"Families with CPC codes: {family_cpc.docdb_family_id.nunique():,} of {len(families):,}")
family_cpc.subclass.value_counts().head(10).to_frame("families")

## 4. Volume

$$\text{Volume}_f=\frac{\text{families of firm } f \text{ in the window}}{\text{all families of the sector in the window}}$$

The denominator includes the families of individuals, universities and small firms.

In [ ]:
volume = (n_families / len(FAMILIES)).reindex(TOP).rename("volume")
firm_bars(volume, "Volume: share of the sector's families, 2010-2017", "share of all families", fmt="{:.1%}")

## 5. Influence

Forward citations come from `TLS228_DOCDB_FAM_CITN` (citing family → cited family). A citation counts if it arrives within 5 years of the first publication of the cited family; its date is the first publication year of the citing family. A family citing itself is dropped. **Self-citations are excluded** (`EXCLUDE_SELF`): a citation is dropped when the citing and the cited family share an owner. The owners are the applicants of PATSTAT (`TLS207`, `applt_seq_nr > 0`), compared by harmonised name `psn_name` (with the group merges of Section 2) and by PATSTAT person id `psn_id`. A citing family of another firm, or one with no applicant in PATSTAT, still counts.

**Normalisation.** The citation count of a family is divided by the mean of the sector families with the **same CPC subclass and the same priority year**. A family with several subclasses gets the mean of its ratios over its subclasses. If a subclass × year cell has fewer than `MIN_COHORT` families, the mean of the year alone is used. Families without CPC codes are compared with the year alone.

$$\text{fwd}^{norm}_{i}=\frac{1}{|S_i|}\sum_{s\in S_i}\frac{\text{fwd}_i}{\bar F_{s,t}}$$

In [ ]:
q = (db.query(TLS228_DOCDB_FAM_CITN.docdb_family_id.label("citing_family"),
              TLS228_DOCDB_FAM_CITN.cited_docdb_family_id.label("cited_family"))
     .filter(TLS228_DOCDB_FAM_CITN.cited_docdb_family_id.in_(FAMILIES)))
fcit = patstat.df(q)
fcit = fcit[fcit.citing_family != fcit.cited_family].copy()

q = (db.query(TLS201_APPLN.docdb_family_id, func.min(TLS201_APPLN.earliest_publn_year).label("year"))
     .filter(TLS201_APPLN.docdb_family_id.in_([int(f) for f in fcit.citing_family.unique()]))
     .filter(TLS201_APPLN.earliest_publn_year < 9999)
     .group_by(TLS201_APPLN.docdb_family_id))
fcit["citing_year"] = fcit.citing_family.map(patstat.df(q).set_index("docdb_family_id").year)
fcit["cited_year"] = fcit.cited_family.map(families.set_index("docdb_family_id").first_publn_year)
fcit["lag"] = fcit.citing_year - fcit.cited_year
print(f"{len(fcit):,} citing-cited pairs from {fcit.citing_family.nunique():,} citing families")

In [ ]:
ALL_FAMS = sorted(set(int(f) for f in FAMILIES) | set(int(f) for f in fcit.citing_family.unique()))
parts = []
for i in range(0, len(ALL_FAMS), 5000):
    q = (db.query(TLS201_APPLN.docdb_family_id, TLS206_PERSON.psn_id, TLS206_PERSON.psn_name)
         .join(TLS207_PERS_APPLN, TLS207_PERS_APPLN.appln_id == TLS201_APPLN.appln_id)
         .join(TLS206_PERSON, TLS206_PERSON.person_id == TLS207_PERS_APPLN.person_id)
         .filter(TLS207_PERS_APPLN.applt_seq_nr > 0)
         .filter(TLS201_APPLN.docdb_family_id.in_(ALL_FAMS[i:i + 5000])))
    parts.append(patstat.df(q))
owners = pd.concat(parts).drop_duplicates()

by_name = owners.assign(key=owners.psn_name.fillna("").map(firm_name))[["docdb_family_id", "key"]]
by_id = owners[owners.psn_id > 0].assign(key=lambda d: "id:" + d.psn_id.astype(int).astype(str))[["docdb_family_id", "key"]]
keys = pd.concat([by_name, by_id]).drop_duplicates()
keys = keys[keys.key != ""]

pairs = fcit[["citing_family", "cited_family"]].drop_duplicates()
same_owner = (pairs.merge(keys.rename(columns={"docdb_family_id": "citing_family"}), on="citing_family")
                   .merge(keys[keys.docdb_family_id.isin(FAMILIES)].rename(columns={"docdb_family_id": "cited_family"}),
                          on=["cited_family", "key"])[["citing_family", "cited_family"]].drop_duplicates())
fcit["self"] = pd.MultiIndex.from_frame(fcit[["citing_family", "cited_family"]]).isin(
    pd.MultiIndex.from_frame(same_owner))
print(f"Self-citations: {fcit['self'].mean():.1%} of the {len(fcit):,} citing-cited pairs")
fcit_all = fcit.copy()                      # with self-citations, for the comparison below
if EXCLUDE_SELF:
    fcit = fcit[~fcit["self"]].copy()
    print(f"Pairs kept: {len(fcit):,}")

In [ ]:
def forward_citations(w, cit=None):
    """Citations within w years of first publication; missing if the window is not complete."""
    cit = fcit if cit is None else cit
    counts = cit[cit.lag.between(0, w)].groupby("cited_family").citing_family.nunique()
    fwd = families.docdb_family_id.map(counts).fillna(0)
    complete = families.first_publn_year + w <= LAST_YEAR
    return fwd.where(complete)


def normalise(values):
    """Ratio to the mean of the same CPC subclass x priority year (fallback: the year alone)."""
    d = pd.DataFrame({"docdb_family_id": families.docdb_family_id, "year": families.priority_year, "v": values})
    year_mean = d.groupby("year").v.transform("mean")
    d["ratio_year"] = (d.v / year_mean).where(year_mean > 0, 0.0)

    cells = family_cpc.merge(d, on="docdb_family_id")
    cells["cell_mean"] = cells.groupby(["subclass", "year"]).v.transform("mean")
    cells["cell_n"] = cells.groupby(["subclass", "year"]).v.transform("count")
    ok = (cells.cell_n >= MIN_COHORT) & (cells.cell_mean > 0)
    cells["ratio"] = np.where(ok, cells.v / cells.cell_mean, np.nan)
    by_family = cells.groupby("docdb_family_id").ratio.mean()

    out = d.docdb_family_id.map(by_family)
    out = out.fillna(d.ratio_year)            # no usable subclass cell: the year alone
    return out.where(d.v.notna()).values


families["fwd5"] = forward_citations(CIT_YEARS)
families["fwd5_norm"] = normalise(families.fwd5)
print("Families with a complete window:", f"{families.fwd5.notna().mean():.0%}")
families.groupby("priority_year").fwd5.mean().round(2).to_frame("mean fwd5 per year").T

In [ ]:
influence = per_firm("fwd5_norm").rename("influence")
firm_bars(influence, "Influence: forward citations within 5 years, relative to the same subclass and year",
          "mean of normalised citations (ratio to the same subclass x year)",
          ref=families.fwd5_norm.mean(), ref_label="sector mean")

### 5.3 Effect of the self-citations

Influence with and without the citations between families of the same firm, and the share of the citations that are self-citations (all years of the window). A firm with a high share owes part of its Influence to its own later patents.

In [ ]:
fwd_all = forward_citations(CIT_YEARS, fcit_all)
families["_fwd5_with_self"] = normalise(fwd_all)
influence_with_self = per_firm("_fwd5_with_self").rename("with self-citations")
cit5 = fcit_all[fcit_all.lag.between(0, CIT_YEARS)].merge(fam_firm, left_on="cited_family", right_on="docdb_family_id")
self_share = cit5.groupby("firm")["self"].mean().reindex(TOP).rename("self-citation share")
compare = pd.concat([influence.rename("without self-citations"), influence_with_self, self_share], axis=1)
compare["rank without"] = compare["without self-citations"].rank(ascending=False).astype(int)
compare["rank with"] = compare["with self-citations"].rank(ascending=False).astype(int)
compare.rename(index=SHORT).round(2)

## 6. Success

Success asks whether the examiners accept the firm's inventions. The sector families are filed worldwide and Japanese firms file mostly in Japan, so a measure on EP applications alone would miss part of the portfolio. We use the **family-level measure**: for each family, the share of its national applications that are granted; then the mean over the firm's families.

Left out: `WO` applications (a PCT request is not a national application) and applications of kind `T` (translations of granted EP patents, always granted). `N` in `granted` mixes refused, withdrawn and pending applications; with priority years up to 2017 most have been decided.

An EP-only version is computed as a check.

In [ ]:
q = (db.query(TLS201_APPLN.appln_id, TLS201_APPLN.docdb_family_id, TLS201_APPLN.appln_auth,
              TLS201_APPLN.appln_kind, TLS201_APPLN.granted)
     .filter(TLS201_APPLN.docdb_family_id.in_(FAMILIES)))
members = patstat.df(q).drop_duplicates("appln_id")
national = members[(members.appln_auth != "WO") & (members.appln_kind != "T")].copy()
national["is_granted"] = (national.granted == "Y").astype(int)

office_rate = national.groupby("appln_auth").agg(applications=("appln_id", "size"), granted=("is_granted", "mean"))
display(office_rate[office_rate.applications >= 50].sort_values("granted", ascending=False).round(2).T)

In [ ]:
families["family_success"] = families.docdb_family_id.map(national.groupby("docdb_family_id").is_granted.mean())
success = per_firm("family_success").rename("success")

ep = national[national.appln_auth == "EP"]
families["ep_success"] = families.docdb_family_id.map(ep.groupby("docdb_family_id").is_granted.mean())
ep_success = per_firm("ep_success").rename("success_ep_only")

print(f"Families with no national application (only WO): {families.family_success.isna().sum()}")
firm_bars(success, "Success: share of national applications granted, per family",
          "mean share granted", fmt="{:.0%}", ref=families.family_success.mean(), ref_label="sector average")
pd.concat([success, ep_success, per_firm("ep_success", "count").rename("families with EP")], axis=1).rename(index=SHORT).round(2)

## 7. Globalization

For each family, the number of distinct **markets** with a national application, among the six markets of the Step 2 PEST: US, Canada, EU, Brazil, China and Japan. The PEST keeps the US and Canada (about 61% of Deere's fiscal 2025 sales), the EU and Brazil as markets, and adds China and Japan because the competitors in our data are based there. Korea, India, Russia and Australia are left out, as in the PEST.

- `EU` is the EP office or the national office of an EU member state.
- `WO` applications (a PCT request) and applications of kind `T` (translations of EP patents) are left out, as in Section 6.
- A family with no application in these markets counts 0.

$$\text{Globalization}_f=\text{mean number of PEST markets per family of firm } f$$

The plain count of distinct offices (the measure of the course notebook) and the share of **quadrilateral families** (CN, EP, JP, US) are computed as checks.

In [ ]:
EU_OFFICES = {"EP", "AT", "BE", "BG", "CY", "CZ", "DE", "DK", "EE", "ES", "FI", "FR", "GR", "HR", "HU", "IE",
              "IT", "LT", "LU", "LV", "MT", "NL", "PL", "PT", "RO", "SE", "SI", "SK"}
MARKETS = ["US", "CA", "EU", "BR", "CN", "JP"]          # from the Step 2 PEST


def market_of(office):
    if office in EU_OFFICES:
        return "EU"
    return office if office in MARKETS else None


national["market"] = national.appln_auth.map(market_of)
families["markets"] = families.docdb_family_id.map(
    national.dropna(subset=["market"]).groupby("docdb_family_id").market.nunique()).fillna(0)
globalization = per_firm("markets").rename("globalization")

families["offices"] = families.docdb_family_id.map(national.groupby("docdb_family_id").appln_auth.nunique()).fillna(0)
globalization_offices = per_firm("offices").rename("globalization_offices")

QUAD = {"CN", "EP", "JP", "US"}
quad = national.groupby("docdb_family_id").appln_auth.apply(lambda s: QUAD.issubset(set(s)))
families["quadrilateral"] = families.docdb_family_id.map(quad).fillna(False).astype(int)
quadrilateral_share = per_firm("quadrilateral").rename("quadrilateral_share")

share_by_market = national.dropna(subset=["market"]).drop_duplicates(["docdb_family_id", "market"]).market.value_counts() / len(families)
print("Share of the families with an application in each market:")
print(share_by_market.round(2).to_dict())
firm_bars(globalization, "Globalization: PEST markets where the inventions are filed",
          "mean number of markets per family (maximum 6)", ref=families.markets.mean(), ref_label="sector average")

## 8. The four indicators together

The indicators are made comparable as in the course notebook: either as **ratio to the mean firm** or as **ranks** combined with Reciprocal Rank Fusion (Cormack et al., 2009). The base case gives the four indicators equal weight. The other schemes test whether the ranking depends on that choice. The reasoning is in the document "Step 3 – Present-value score: weighting decision (team 38)".

The schemes, and the rule for the final choice, were fixed before reading the results.

In [ ]:
INDICATORS = ["volume", "influence", "success", "globalization"]
firms = pd.concat([n_families.reindex(TOP).rename("families"), volume, influence, success, globalization], axis=1)
extras = pd.concat([globalization_offices, quadrilateral_share, ep_success], axis=1)

print("Missing values:", firms[INDICATORS].isna().sum().to_dict())
firms.rename(index=SHORT).round(3)

In [ ]:
SCHEMES = {
    "base: equal": {k: 1 for k in INDICATORS},
    "Clarivate-style": {"influence": 1, "success": 1, "globalization": 1},
    "impact first": {"volume": 0.20, "influence": 0.40, "success": 0.20, "globalization": 0.20},
    "size first": {"volume": 0.40, "influence": 0.20, "success": 0.20, "globalization": 0.20},
}
for k in INDICATORS:
    SCHEMES[f"without {k}"] = {j: 1 for j in INDICATORS if j != k}
BASE = "base: equal"
RRF_C = len(TOP)                  # constant of RRF: the number of firms


def combine(table, weights, method):
    """One score per firm, higher is better. 'mean': 100 = average firm. 'rrf': 100 = first on every indicator."""
    w = pd.Series(weights, dtype=float)
    w = w / w.sum()
    t = table[w.index]
    if method == "mean":
        return 100 * (t / t.mean() * w).sum(axis=1)
    if method == "rrf":
        pos = t.rank(ascending=False, method="min")
        return 100 * (w * (RRF_C + 1) / (RRF_C + pos)).sum(axis=1)
    raise ValueError("method must be 'mean' or 'rrf'")


def final_rank(scores):
    return scores.rank(ascending=False, method="min").astype(int)


scores = {m: pd.DataFrame({n: combine(firms, w, m) for n, w in SCHEMES.items()}) for m in ["rrf", "mean"]}
ranks = {m: scores[m].apply(final_rank) for m in scores}

### 8.1 Ranking under each scheme (1 = best)

In [ ]:
for m, label in [("rrf", "Ranks combined with RRF"), ("mean", "Ratio to the mean firm")]:
    print(label)
    display(ranks[m].rename(index=SHORT).sort_values(BASE))

### 8.2 How much do the schemes agree?

Rank correlation (Spearman, 1 = same order) of each scheme with the base case, and the best and worst position of each firm across all schemes and both methods. A firm with a narrow range is a robust result; a firm with a wide range owes its position to the weights.

In [ ]:
agree = pd.DataFrame({m: ranks[m].corr(method="spearman")[BASE].drop(BASE) for m in ranks}).round(2)
agree.columns = ["RRF", "mean"]
display(agree)

all_ranks = pd.concat([ranks["rrf"], ranks["mean"]], axis=1)
spread = pd.DataFrame({"base (RRF)": ranks["rrf"][BASE], "best": all_ranks.min(axis=1), "worst": all_ranks.max(axis=1)})
spread["range"] = spread.worst - spread.best
spread.rename(index=SHORT).sort_values("base (RRF)")

In [ ]:
METHOD = "rrf"                    # headline method: "rrf" or "mean"
firms["score"] = scores[METHOD][BASE]
firms["rank"] = ranks[METHOD][BASE]
firms.sort_values("rank")[["families", "score", "rank"]].round(1).rename(index=SHORT)

### 8.3 Check of a single firm

For a firm with an unexpected value (for example a very low Success), the national applications of its families by office, with how many are granted. A low rate can come from applications still pending, withdrawn applications, or offices where the `granted` flag is incomplete.

In [ ]:
def check_firm(name):
    ids = fam_firm.loc[fam_firm.firm == name, "docdb_family_id"]
    sub = national[national.docdb_family_id.isin(ids)]
    out = sub.groupby("appln_auth").agg(applications=("appln_id", "size"), granted=("is_granted", "sum"))
    out["rate"] = (out.granted / out.applications).round(2)
    return out.sort_values("applications", ascending=False)


check_firm(TOP[-1])        # change the name to inspect another firm, e.g. check_firm("BOSCH")

## 9. Output for Step 5

In [ ]:
result = firms.join(extras)
for name in SCHEMES:
    result[f"score_{name}"] = scores[METHOD][name]
    result[f"rank_{name}"] = ranks[METHOD][name]
result = result.sort_values("rank")
result.index.name = "firm"
result.to_csv(OUT / "present_score.csv")
families.to_csv(OUT / "present_score_families.csv", index=False)
print(f"Saved {len(result)} firms to {OUT / 'present_score.csv'}")
result.rename(index=SHORT).round(3)

## 10. To do

- Window: ends in 2017 because the grant rate of 2018 is clearly lower (0.62 against 0.70-0.73): the 2018 applications are still being examined.
- Threshold: 20 families (25 would leave out CLAAS, 24 families in 2010-2017).
- Check the lowest Success values with `check_firm` (Section 8.3) before using them in the deck.
- Refinements to consider: share of the sector over time, 3 and 7-year citation windows, blocking patents (X and I citations), uncertainty of each score by resampling families.
- Check `LAST_YEAR` against the PATSTAT edition on TIP.
- Compare the firm list with the Step 2 top-applicant table.